#  Week 2, Day 6 — May 30, 2026
## Schema Alignment Audit & Cleaning Threshold Documentation



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {
    "raw":            BASE_DIR + "/data/raw",
    "processed":      BASE_DIR + "/data/processed",
    "metadata":       BASE_DIR + "/data/metadata",
    "notebooks":      BASE_DIR + "/notebooks",
    "visualizations": BASE_DIR + "/visualizations"
}
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 5, 30, 65, 95
YEAR_MIN, YEAR_MAX = 2015, 2026

In [ ]:
df_p = pd.read_csv(DIRS['processed'] + '/plastic_clean_final.csv', parse_dates=['Date'])
df_s = pd.read_csv(DIRS['processed'] + '/species_clean_final.csv')
print('Loaded final cleaned datasets ')

In [ ]:
# Verify join key compatibility
print('JOIN KEY COMPATIBILITY CHECK')
print('='*55)
print()
print('Planned join keys for Master Table (Week 3):')
print('  Primary: grid_lat (int), grid_lon (int)')
print('  Secondary: year (int), month (int)')
print()

# Check plastic
df_p['Date_parsed'] = pd.to_datetime(df_p['Date'])
df_p['year']  = df_p['Date_parsed'].dt.year.astype(int)
df_p['month'] = df_p['Date_parsed'].dt.month.astype(int)
df_p['grid_lat'] = df_p['Latitude'].apply(lambda x: int(x))
df_p['grid_lon'] = df_p['Longitude'].apply(lambda x: int(x))

print(f'Plastic join keys:')
print(f'  year  : {df_p["year"].dtype}  range {df_p["year"].min()}–{df_p["year"].max()}')
print(f'  month : {df_p["month"].dtype}  range {df_p["month"].min()}–{df_p["month"].max()}')
print(f'  grid_lat: {df_p["grid_lat"].dtype}  range {df_p["grid_lat"].min()}–{df_p["grid_lat"].max()}')
print(f'  grid_lon: {df_p["grid_lon"].dtype}  range {df_p["grid_lon"].min()}–{df_p["grid_lon"].max()}')
print()

import numpy as np
df_s['grid_lat'] = df_s['latitude'].apply(lambda x: int(np.floor(x)))
df_s['grid_lon'] = df_s['longitude'].apply(lambda x: int(np.floor(x)))

print(f'Species join keys:')
print(f'  year  : {df_s["year"].dtype}  range {df_s["year"].min()}–{df_s["year"].max()}')
print(f'  month : {df_s["month"].dtype}  range {df_s["month"].min()}–{df_s["month"].max()}')
print(f'  grid_lat: {df_s["grid_lat"].dtype}  range {df_s["grid_lat"].min()}–{df_s["grid_lat"].max()}')
print(f'  grid_lon: {df_s["grid_lon"].dtype}  range {df_s["grid_lon"].min()}–{df_s["grid_lon"].max()}')

# Check overlapping grid cells
plastic_cells  = set(zip(df_p['grid_lat'], df_p['grid_lon']))
species_cells  = set(zip(df_s['grid_lat'], df_s['grid_lon']))
overlap = plastic_cells.intersection(species_cells)
print(f'\nOverlapping grid cells: {len(overlap)} / {len(plastic_cells)} plastic cells')
print('✅ Overlap confirmed — spatial join will produce matches' if len(overlap) > 0 else '⚠️ No overlap!')

In [ ]:
# Save schema alignment doc
doc_path = DIRS['metadata'] + '/schema_alignment_may30.txt'
with open(doc_path, 'w') as f:
    f.write('Schema Alignment & Cleaning Threshold Documentation\n')
    f.write('May 30, 2026\n')
    f.write('='*60 + '\n\n')
    f.write('CLEANING THRESHOLDS APPLIED:\n')
    thresholds = [
        ('Null drop threshold',         '> 50% null → DROP column'),
        ('Null impute (numeric)',        'median value, per column'),
        ('Null impute (categorical)',    'fill with string "Unknown"'),
        ('Year filter',                  '2015 <= year <= 2026'),
        ('Bounding box',                 'Lat 5N–30N, Lon 65E–95E'),
        ('IQR outlier treatment',        'Clip to [Q1-1.5IQR, Q3+1.5IQR] if outliers>0'),
        ('Plastic unit conversion',      'Plastic_Weight_kg / Cell_Area_km2 = Density_kg_km2'),
        ('Plastic_Type normalization',   'Full names → PET, PE, PS, PP, PVC'),
        ('observation_type normalize',   '12 labels → 5 categories'),
        ('SST units',                    'Confirmed Celsius — no conversion'),
    ]
    for threshold, rule in thresholds:
        f.write(f'  {threshold:<35}: {rule}\n')
    f.write('\nJOIN KEYS FOR MASTER TABLE:\n')
    f.write('  grid_lat (int) = floor(Latitude)\n')
    f.write('  grid_lon (int) = floor(Longitude)\n')
    f.write('  year     (int)\n')
    f.write('  month    (int)\n')
print(f'Schema alignment doc saved ')